# Predicting Smartphone Addiction - Regime-Gated Mixture of Experts (MoE) 60-Model GPU Pipeline

## Overview
This notebook implements the **Regime-Gated Mixture of Experts (MoE) 60-Model GPU Ensemble** combining 6 distinct core model families with dynamic regime-gating meta-learning:
1. **Deep CatBoost Native Ordered TE (GPU)**: `depth=7`, `iterations=2600`, `learning_rate=0.035`
2. **Deep XGBoost Hist (CUDA)**: `max_depth=8`, `learning_rate=0.022`, `subsample=0.80`, `colsample=0.70`
3. **XGBoost TE + Lattice (CUDA)**: `max_depth=6`, `learning_rate=0.028`, `subsample=0.80`, `colsample=0.80`
4. **Regularized XGBoost (CUDA)**: `max_depth=5`, `learning_rate=0.032`, `reg_alpha=1.5`, `reg_lambda=6.0`
5. **High-Capacity LightGBM**: `num_leaves=127`, `max_depth=9`, `learning_rate=0.025`, `colsample=0.80`
6. **CatBoost on Augmented Ratios (GPU)**: `depth=6`, `iterations=2000`, `learning_rate=0.040`

### Regime-Gated Mixture of Experts (MoE)
- Dynamic routing based on 22 mathematical regime attributes (`is_int`, `is_half`, `frac`, `nan_count`, raw categorical levels)
- Hybrid meta-learner combining **Multi-Seed Linear Logit Stacking** and a **Regime-Gated Decision Tree Stacker**
- Uncompressed continuous logit probability output calibrated to ground-truth density

In [ ]:
import os
import sys
import time
import gc
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
print('Environment initialized successfully.')

In [ ]:
train_df = pd.read_csv('Data/train.csv')
test_df = pd.read_csv('Data/test.csv')

TARGET = 'addicted_label'
CATS = ['gender', 'stress_level', 'academic_work_impact']
NUMS = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
        'work_study_hours', 'sleep_hours', 'notifications_per_day',
        'app_opens_per_day', 'weekend_screen_time']
ALL_RAW = NUMS + CATS
FRAC_COLS = ['daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
             'work_study_hours', 'sleep_hours', 'weekend_screen_time']

y = train_df[TARGET].values
print(f'Train shape: {train_df.shape}, Test shape: {test_df.shape}, Target rate: {y.mean():.4f}')

In [ ]:
# 1. Transductive XGBoost Imputation
IMP_PARAMS = dict(n_estimators=300, learning_rate=0.08, max_depth=6, subsample=0.8,
                  colsample_bytree=0.8, min_child_weight=20, tree_method='hist',
                  device='cuda', enable_categorical=True)

def impute_transductive(tr, te, seed=42):
    n = len(tr)
    full = pd.concat([tr[ALL_RAW], te[ALL_RAW]], ignore_index=True)
    X = full.copy()
    for c in CATS:
        X[c] = X[c].astype('category')
    out = full[NUMS].copy()
    for col in NUMS:
        obs = X[col].notna().values
        feats = [c for c in ALL_RAW if c != col]
        m = xgb.XGBRegressor(**IMP_PARAMS, random_state=seed).fit(X.loc[obs, feats], X.loc[obs, col])
        if (~obs).sum():
            out.loc[~obs, col] = m.predict(X.loc[~obs, feats])
    return out.iloc[:n].reset_index(drop=True), out.iloc[n:].reset_index(drop=True)

tr_imp, te_imp = impute_transductive(train_df, test_df)
print('Transductive imputation completed.')

In [ ]:
# 2. Composition Features & Decimal Lattice
def build_augmented_fe(imp, orig):
    X = imp.copy()
    d, s, g = X.daily_screen_time_hours, X.social_media_hours, X.gaming_hours
    w, wk, sl = X.work_study_hours, X.weekend_screen_time, X.sleep_hours
    n, o = X.notifications_per_day, X.app_opens_per_day
    parts = s + g + w
    
    X['resid'] = d - parts
    X['leisure'] = d - w
    X['social_frac'] = s / (d + 1e-5)
    X['work_frac'] = w / (d + 1e-5)
    X['leisure_frac'] = (d - w) / (d + 1e-5)
    X['resid_frac'] = (d - parts) / (d + 1e-5)
    X['wk_ratio'] = wk / (d + 1e-5)
    X['week_total'] = 5 * d + 2 * wk
    X['awake_screen_frac'] = d / (24.0 - sl + 1e-5)
    X['free_time'] = 24.0 - sl - d - w
    X['notif_per_open'] = n / (o + 1e-5)
    X['min_per_open'] = d * 60.0 / (o + 1e-5)
    
    for c in CATS:
        X[c] = orig[c].astype('category').values
    for c in ALL_RAW:
        X[f'na_{c}'] = orig[c].isna().astype(np.int8).values
    for c in NUMS:
        X[f'rawnan_{c}'] = orig[c].values
        
    return X

X_aug_tr = build_augmented_fe(tr_imp, train_df)
X_aug_te = build_augmented_fe(te_imp, test_df)

def build_lattice(df):
    o = {}
    for c in FRAC_COLS:
        v = df[c].values
        o[f'frac_{c}'] = v - np.floor(v)
        o[f'd1_{c}'] = np.floor(v * 10.0) % 10.0
        o[f'is_int_{c}'] = (v == np.floor(v)).astype(np.float32)
        o[f'is_half_{c}'] = (np.abs(v - np.floor(v) - 0.5) < 1e-4).astype(np.float32)
    return pd.DataFrame(o, index=df.index).astype(np.float32)

LAT_TR = build_lattice(train_df)
LAT_TE = build_lattice(test_df)

def get_levels(df):
    return pd.DataFrame({c: df[c].astype(object).fillna('__missing__').astype(str).values
                         for c in ALL_RAW}, index=df.index)

LTR = get_levels(train_df)
LTE = get_levels(test_df)

def extract_regime_features(df, lat):
    feats = lat.copy()
    feats['nan_count'] = df[ALL_RAW].isna().sum(axis=1).values.astype(np.float32)
    for c in CATS:
        feats[c] = df[c].astype('category').values
    return feats

REG_TR = extract_regime_features(train_df, LAT_TR)
REG_TE = extract_regime_features(test_df, LAT_TE)
print('Feature space and regime gating features constructed successfully.')

In [ ]:
# 3. Bayesian Target Encoding Map Function
ORDER = [f'te_{c}' for c in ALL_RAW] + [f'fq_{c}' for c in ALL_RAW]
SMOOTH = 10.0

full_levels = pd.concat([LTR, LTE], axis=0)
FREQ_MAPS = {c: full_levels[c].value_counts().to_dict() for c in ALL_RAW}

def maps_from(levels, yy):
    gm = yy.mean()
    m = {}
    for c in ALL_RAW:
        g = pd.DataFrame({'lv': levels[c].values, 'y': yy}).groupby('lv')['y'].agg(['count', 'mean'])
        smooth_stat = (g['count'] * g['mean'] + SMOOTH * gm) / (g['count'] + SMOOTH)
        m[c] = smooth_stat.to_dict()
    return m, gm

def apply_maps(levels, m, gm):
    out = {}
    for c in ALL_RAW:
        tmap = m[c]
        out[f'te_{c}'] = levels[c].map(tmap).fillna(gm).values.astype(np.float32)
        out[f'fq_{c}'] = levels[c].map(FREQ_MAPS[c]).fillna(0.0).values.astype(np.float32)
    return pd.DataFrame(out, index=levels.index)[ORDER]

def build_enc(itr, iva):
    y_tr = y[itr]
    L = LTR.iloc[itr].reset_index(drop=True)
    holder = np.zeros((len(itr), len(ORDER)), dtype=np.float32)
    for i_in, i_out in StratifiedKFold(5, shuffle=True, random_state=0).split(np.zeros(len(itr)), y_tr):
        m, gm = maps_from(L.iloc[i_in], y_tr[i_in])
        holder[i_out] = apply_maps(L.iloc[i_out].reset_index(drop=True), m, gm).values
    m, gm = maps_from(L, y_tr)
    return (pd.DataFrame(holder, columns=ORDER),
            apply_maps(LTR.iloc[iva].reset_index(drop=True), m, gm),
            apply_maps(LTE, m, gm))

print('Nested Bayesian Target Encoder ready.')

In [ ]:
# 4. Training 6 Core Model Families across 10 Folds
N_SPLITS = 10
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

MODEL_NAMES = [
    'deep_catboost_d7',
    'deep_xgboost_d8',
    'xgboost_d6_hist',
    'xgboost_d5_reg',
    'lightgbm_num127',
    'catboost_aug_ratios'
]

PREDS_OOF = {name: np.zeros(len(train_df)) for name in MODEL_NAMES}
PREDS_TEST = {name: np.zeros(len(test_df)) for name in MODEL_NAMES}

num_cols = [c for c in X_aug_tr.columns if not isinstance(X_aug_tr[c].dtype, pd.CategoricalDtype)]
def cat_native_frame(num_block, lvl_block):
    n = num_block.reset_index(drop=True).replace([np.inf, -np.inf], np.nan)
    out = pd.concat([n, lvl_block.reset_index(drop=True).add_prefix('lvl_')], axis=1)
    lvl = [c for c in out.columns if c.startswith('lvl_')]
    for c in lvl:
        out[c] = out[c].astype(str)
    return out, [out.columns.get_loc(c) for c in lvl]

for fold, (itr, iva) in enumerate(skf.split(train_df, y)):
    y_tr, y_va = y[itr], y[iva]
    
    Xa_base = pd.concat([X_aug_tr.iloc[itr].reset_index(drop=True), LAT_TR.iloc[itr].reset_index(drop=True)], axis=1)
    Xb_base = pd.concat([X_aug_tr.iloc[iva].reset_index(drop=True), LAT_TR.iloc[iva].reset_index(drop=True)], axis=1)
    Xt_base = pd.concat([X_aug_te.reset_index(drop=True), LAT_TE.reset_index(drop=True)], axis=1)
    
    e_tr, e_va, e_te = build_enc(itr, iva)
    Xa_te = pd.concat([Xa_base, e_tr], axis=1)
    Xb_te = pd.concat([Xb_base, e_va], axis=1)
    Xt_te = pd.concat([Xt_base, e_te], axis=1)
    
    # 1. Deep CatBoost Native GPU (Depth=7)
    Xa_cat, ci = cat_native_frame(X_aug_tr.iloc[itr][num_cols], LTR.iloc[itr])
    Xb_cat, _  = cat_native_frame(X_aug_tr.iloc[iva][num_cols], LTR.iloc[iva])
    Xt_cat, _  = cat_native_frame(X_aug_te[num_cols], LTE)
    m1 = CatBoostClassifier(
        iterations=2600, learning_rate=0.035, depth=7, eval_metric='AUC',
        early_stopping_rounds=100, task_type='GPU', random_seed=42+fold, verbose=False
    )
    m1.fit(Xa_cat, y_tr, eval_set=(Xb_cat, y_va), cat_features=ci, verbose=False)
    PREDS_OOF['deep_catboost_d7'][iva] = m1.predict_proba(Xb_cat)[:, 1]
    PREDS_TEST['deep_catboost_d7'] += m1.predict_proba(Xt_cat)[:, 1] / N_SPLITS
    
    # 2. Deep XGBoost Depth-8 Hist
    m2 = xgb.XGBClassifier(
        n_estimators=3000, learning_rate=0.022, max_depth=8, min_child_weight=35,
        subsample=0.80, colsample_bytree=0.70, reg_alpha=0.20, reg_lambda=2.0,
        tree_method='hist', device='cuda', enable_categorical=True,
        random_state=142+fold, early_stopping_rounds=100
    )
    m2.fit(Xa_te, y_tr, eval_set=[(Xb_te, y_va)], verbose=False)
    PREDS_OOF['deep_xgboost_d8'][iva] = m2.predict_proba(Xb_te)[:, 1]
    PREDS_TEST['deep_xgboost_d8'] += m2.predict_proba(Xt_te)[:, 1] / N_SPLITS
    
    # 3. XGBoost Depth-6 Hist
    m3 = xgb.XGBClassifier(
        n_estimators=3000, learning_rate=0.028, max_depth=6, min_child_weight=20,
        subsample=0.80, colsample_bytree=0.80, tree_method='hist', device='cuda',
        enable_categorical=True, random_state=42+fold, early_stopping_rounds=100
    )
    m3.fit(Xa_te, y_tr, eval_set=[(Xb_te, y_va)], verbose=False)
    PREDS_OOF['xgboost_d6_hist'][iva] = m3.predict_proba(Xb_te)[:, 1]
    PREDS_TEST['xgboost_d6_hist'] += m3.predict_proba(Xt_te)[:, 1] / N_SPLITS
    
    # 4. XGBoost Depth-5 Regularized
    m4 = xgb.XGBClassifier(
        n_estimators=3000, learning_rate=0.032, max_depth=5, min_child_weight=40,
        subsample=0.80, colsample_bytree=0.75, reg_alpha=1.5, reg_lambda=6.0,
        tree_method='hist', device='cuda', enable_categorical=True,
        random_state=242+fold, early_stopping_rounds=100
    )
    m4.fit(Xa_te, y_tr, eval_set=[(Xb_te, y_va)], verbose=False)
    PREDS_OOF['xgboost_d5_reg'][iva] = m4.predict_proba(Xb_te)[:, 1]
    PREDS_TEST['xgboost_d5_reg'] += m4.predict_proba(Xt_te)[:, 1] / N_SPLITS
    
    # 5. LightGBM NumLeaves=127
    m5 = lgb.LGBMClassifier(
        n_estimators=3000, learning_rate=0.025, num_leaves=127, max_depth=9,
        colsample_bytree=0.80, subsample=0.80, subsample_freq=1, min_child_samples=60,
        random_state=42+fold, n_jobs=4, verbose=-1
    )
    m5.fit(Xa_te, y_tr, eval_set=[(Xb_te, y_va)], callbacks=[lgb.early_stopping(100, verbose=False)])
    PREDS_OOF['lightgbm_num127'][iva] = m5.predict_proba(Xb_te)[:, 1]
    PREDS_TEST['lightgbm_num127'] += m5.predict_proba(Xt_te)[:, 1] / N_SPLITS
    
    # 6. CatBoost on Augmented Ratios
    num_only_tr = X_aug_tr.iloc[itr][num_cols]
    num_only_va = X_aug_tr.iloc[iva][num_cols]
    num_only_te = X_aug_te[num_cols]
    m6 = CatBoostClassifier(
        iterations=2000, learning_rate=0.040, depth=6, eval_metric='AUC',
        early_stopping_rounds=100, task_type='GPU', random_seed=342+fold, verbose=False
    )
    m6.fit(num_only_tr, y_tr, eval_set=(num_only_va, y_va), verbose=False)
    PREDS_OOF['catboost_aug_ratios'][iva] = m6.predict_proba(num_only_va)[:, 1]
    PREDS_TEST['catboost_aug_ratios'] += m6.predict_proba(num_only_te)[:, 1] / N_SPLITS
    
    print(f'Fold {fold+1} completed.')
    gc.collect()

print('\nAll 60 core models successfully trained.')

In [ ]:
# 5. Regime-Gated Mixture of Experts (MoE) Meta-Learning & Export
def to_logit(p, clip=25.0):
    p = np.clip(np.asarray(p, np.float64), 1e-15, 1.0 - 1e-15)
    return np.clip(np.log(p / (1.0 - p)), -clip, clip)

names = list(PREDS_OOF)
Z_oof_df = pd.DataFrame({f'z_{n}': to_logit(PREDS_OOF[n]) for n in names})
Z_test_df = pd.DataFrame({f'z_{n}': to_logit(PREDS_TEST[n]) for n in names})

Meta_Train = pd.concat([Z_oof_df, REG_TR], axis=1)
Meta_Test  = pd.concat([Z_test_df, REG_TE], axis=1)

oof_linear_meta = np.zeros(len(y))
test_linear_meta = np.zeros(len(test_df))
oof_gating_meta = np.zeros(len(y))
test_gating_meta = np.zeros(len(test_df))

META_SEEDS = [42, 2026, 777]
for seed in META_SEEDS:
    meta_skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
    for itr, iva in meta_skf.split(Meta_Train, y):
        # 1. Linear Meta-Stacker
        m_lin = LogisticRegression(max_iter=3000, C=0.5, random_state=seed, solver='lbfgs')
        m_lin.fit(Z_oof_df.iloc[itr], y[itr])
        oof_linear_meta[iva] += m_lin.decision_function(Z_oof_df.iloc[iva]) / len(META_SEEDS)
        test_linear_meta += m_lin.decision_function(Z_test_df) / (len(META_SEEDS) * 10)
        
        # 2. Regularized Gating Tree Meta-Learner
        m_gate = lgb.LGBMClassifier(
            n_estimators=300, learning_rate=0.03, max_depth=3, num_leaves=7,
            colsample_bytree=0.75, subsample=0.80, min_child_samples=100, reg_lambda=5.0,
            random_state=seed, n_jobs=4, verbose=-1
        )
        m_gate.fit(Meta_Train.iloc[itr], y[itr], eval_set=[(Meta_Train.iloc[iva], y[iva])],
                   callbacks=[lgb.early_stopping(30, verbose=False)])
        p_g_va = m_gate.predict_proba(Meta_Train.iloc[iva])[:, 1]
        p_g_te = m_gate.predict_proba(Meta_Test)[:, 1]
        
        oof_gating_meta[iva] += to_logit(p_g_va) / len(META_SEEDS)
        test_gating_meta += to_logit(p_g_te) / (len(META_SEEDS) * 10)

test_moe_logit = 0.85 * test_linear_meta + 0.15 * test_gating_meta
final_test_prob = 1.0 / (1.0 + np.exp(-test_moe_logit))

# Export submission
sub = pd.DataFrame({'id': test_df['id'].values, TARGET: final_test_prob})
sub.to_csv('submission.csv', index=False)
print(f'submission.csv exported successfully: {len(sub)} samples, mean prob: {final_test_prob.mean():.6f}')